# GYM AI
AI workout planner and tracker

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3
import base64
from io import BytesIO
from PIL import Image

In [ ]:
load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print(f"OpenAI key is available and starts with {openai_api_key[:8]}")
else:
    print("OpenAI key is not available")

openai = OpenAI()
MODEL = "gpt-4.1-mini"

DB = "workouts.db"

In [ ]:
system_message = """
You are a motivational assistant that plans and tracks gym workouts.
You will be given a workout plan and you will keep track of the user's progress.
You will provide information about the exercises, sets, reps,
and weights that the user should do for a particular day.
If you don't have the information, 
you will ask the user to plan their workout by asking them questions
about what exercise they want to do, how many sets and reps they want to do, 
and what weights they want to use.
You will also provide feedback and suggestions based on the user's performance.
Keep your responses starightforward and concise, and avoid unnecessary explanations.
"""

In [ ]:
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS workouts (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            day TEXT NOT NULL,
            exercise TEXT NOT NULL,
            sets INTEGER NOT NULL,
            reps INTEGER NOT NULL,
            kg_weight REAL NOT NULL
        )
        """
        )
    conn.commit()

In [ ]:
def get_workout(day):
    print(f"DATABASE TOOL CALLED: Getting workout for {day}")
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT exercise, sets, reps, kg_weight FROM workouts WHERE day = ?", (day.lower(),))
        result = cursor.fetchall()
        return f"Workouts for {day}: {result}" if result else None

In [ ]:
def set_workout(day, exercise, sets, reps, kg_weight):
    print(f"DATABASE TOOL CALLED: Setting workout for {day} - {exercise}, {sets} sets of {reps} reps at {kg_weight} kg")
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            "INSERT INTO workouts (day, exercise, sets, reps, kg_weight) VALUES (?, ?, ?, ?, ?)",
            (day.lower(), exercise, sets, reps, kg_weight)
        )
        conn.commit()
        return f"Workout for {day} set: {exercise}, {sets} sets of {reps} reps at {kg_weight} kg was added."

In [ ]:
set_workout("Monday", "Bench Press", 4, 8, 80)

In [ ]:
get_workout_function = {
    "name": "get_workout",
    "description": "Get all workouts to be performed on a particular day",
    "parameters": {
        "type": "object",
        "properties": {
            "day": {
                "type": "string",
                "description": "The day for which to retrieve the workout plan (e.g., 'Monday', 'Tuesday', etc.)",
            },
        },
        "required": ["day"],
        "additionalProperties": False
    }
}

set_workout_function = {
    "name": "set_workout",
    "description": "Set a workout for a prticular day",
    "parameters": {
        "type": "object",
        "properties": {
            "day": {
                "type": "string",
                "description": "The day for which to set the workout plan (e.g., 'Monday', 'Tuesday', etc.)",
            },
            "exercise": {
                "type": "string",
                "description": "The name of the exercise (e.g., 'Bench Press', 'Squats', etc.)",
            },
            "sets": {
                "type": "integer",
                "description": "The number of sets to perform for the exercise.",
            },
            "reps": {
                "type": "integer",
                "description": "The number of repetitions to perform for each set.",
            },
            "kg_weight": {
                "type": "number",
                "description": "The weight in kilograms to be used for the exercise.",
            },
        }
    }
}

tools = [
    {"type": "function", "function": get_workout_function},
    {"type": "function", "function": set_workout_function}
]

In [ ]:
def chat(history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools
    )
    exercises = []
    image = None


    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        responses, exercises = handle_tool_calls(message)
        messages += [message] + responses
        response = openai.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )

    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    if exercises:
        image = create_workout_image(exercises)
    
    return history, image

In [ ]:
chat("What is my workout for Monday?", [])

In [ ]:
def handle_tool_calls(message):
    responses = []
    exercises = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_workout":
            arguments = json.loads(tool_call.function.arguments)
            day = arguments.get("day")
            workout = get_workout(day)
            exercises.extend(workout)
            print(exercises)
            response = {
                "role": "tool",
                "content": json.dumps(workout),
                "tool_call_id": tool_call.id
            }
            responses.append(response)

        if tool_call.function.name == "set_workout":
            arguments = json.loads(tool_call.function.arguments)
            day = arguments.get("day")
            exercise = arguments.get("exercise")
            sets = arguments.get("sets")
            reps = arguments.get("reps")
            kg_weight = arguments.get("kg_weight")
            set_workout_response = set_workout(day, exercise, sets, reps, kg_weight)
            response = {
                "role": "tool",
                "content": json.dumps(set_workout_response),
                "tool_call_id": tool_call.id
            }
            responses.append(response)
    return responses, exercises

In [ ]:
def get_workout_image_prompt(day_workout):
    return f"Create an image of a musclar person doing these exercises: {day_workout} \nMake the image vibrant and motivational, with a gym background. The person should be in action, showcasing strength and determination."

In [ ]:
def create_workout_image(workout):
    image_response = openai.images.generate(
        model="gpt-image-2",
        prompt=get_workout_image_prompt(workout),
        size="1024x1024",
        n=1
    )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

In [ ]:
image = create_workout_image("Bench Press, Squats, Deadlifts")
display(image)

In [ ]:
def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]


with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[
             chatbot,
             image_output
            ]
    )

ui.launch(inbrowser=True)